# 5주차: YOLOv8 vs YOLOv8 + CNN
동일 탐지기의 후보를 이진 CNN으로 재평가합니다. train=학습, val=선택, test=최종 비교.

실제 데이터와 학습 가중치는 Google Drive 경로에 있어야 합니다. 이 노트북에는 가짜 성능 수치를 넣지 않았습니다. GPU 런타임을 선택하고 위에서부터 순서대로 실행하세요.


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "torchmetrics>=1.4,<2" pycocotools opencv-python-headless matplotlib pandas pyyaml
from google.colab import drive
import os
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU: GPU 런타임 권장")


## 1. 경로·공통 조건 설정
기존 가중치가 test를 학습하거나 선택에 사용했으면 새로운 held-out test가 필요합니다. `valid/images`가 실제 폴더인지 확인하세요. 기존 `images_aug`를 YOLO 신규 학습에 쓰려면 별도 데이터 루트에서 표준 `images/labels`로 정리하세요.


In [ ]:
from pathlib import Path
# 기존 YOLO best.pt가 있으면 아래 yolo_weights에 절대경로를 입력하세요.
# 예: /content/drive/MyDrive/Pothole/runs/yolov8s_weather_opt/weights/best.pt
CFG = {
    'data_root': '/content/drive/MyDrive/Pothole',
    'splits': {'train':'train/images', 'val':'valid/images', 'test':'test/images'},
    'groups_csv': '',  # 선택: 영상/도로 group별 split 누수 확인 CSV
    'output_root': '/content/drive/MyDrive/Pothole/week5_results',
    'yolo_weights': '',  # 비어 있으면 YOLOv8s 신규 학습
    'weights_provenance_confirmed': False, # 재사용 시 test 미사용 확인 후 True
    'yolo_init': 'yolov8s.pt', 'yolo_epochs':100, 'yolo_batch':16,
    'seed':42, 'imgsz':800, 'proposal_conf':.001, 'nms_iou':.7, 'max_det':300,
    'crop_size':96, 'cnn_epochs':30, 'cnn_patience':7, 'cnn_batch':64, 'cnn_lr':.001,
    'backgrounds_per_image':12, 'alpha_grid':[0.,.25,.5,1.],
    'shared_threshold':.25, 'bootstrap_n':2000,
    'benchmark_images':30, 'benchmark_repeats':3,
}
# CUDA OOM이면 yolo_batch만 8로 낮추고 전체 실험을 새로 실행하세요.


## 2. 실행 모듈 생성
아래 셀은 비교 코드를 /content에 기록합니다. 직접 추가 파일을 업로드할 필요가 없습니다.


In [ ]:
%%writefile /content/week5_compare.py
"""Week 5: shared YOLOv8 detector + binary CNN proposal rescoring.
Run from the accompanying Colab notebook. No test labels are used for training/tuning.
"""
from pathlib import Path
import copy, csv, hashlib, json, random, time, platform, subprocess, sys
import cv2
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO
from torchmetrics.detection.mean_ap import MeanAveragePrecision


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()


def ious(a, b):
    a=np.asarray(a, dtype=np.float32).reshape(-1,4)
    b=np.asarray(b, dtype=np.float32).reshape(-1,4)
    lo=np.maximum(a[:,None,:2], b[None,:,:2]); hi=np.minimum(a[:,None,2:], b[None,:,2:])
    inter=np.maximum(hi-lo,0).prod(2)
    aa=np.maximum(a[:,2:]-a[:,:2],0).prod(1)
    bb=np.maximum(b[:,2:]-b[:,:2],0).prod(1)
    return inter / np.maximum(aa[:,None]+bb[None,:]-inter, 1e-9)


def audit_dataset(cfg, out):
    """Explicit YOLO directory layout; missing annotation != negative image."""
    root=Path(cfg['data_root']); records={}; fingerprints={}; manifest=[]
    for split, rel in cfg['splits'].items():
        images=root/rel
        labels=images.parent/'labels' if images.name=='images' else Path(str(images).replace('images_aug','labels_aug'))
        if labels == images: raise ValueError('Use split/images or split/images_aug directory layout')
        files=sorted(p for p in images.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'})
        if not files: raise ValueError(f'{split}: 이미지가 없습니다: {images}')
        rows=[]
        for p in files:
            im=cv2.imread(str(p))
            if im is None: raise ValueError(f'손상된 이미지: {p}')
            h,w=im.shape[:2]; label=labels/(p.stem+'.txt')
            if not label.exists(): raise FileNotFoundError(f'라벨 누락: {label}; 배경 이미지는 빈 txt 파일 필요')
            raw=[]
            for line in label.read_text().splitlines():
                if not line.strip(): continue
                v=list(map(float,line.split()))
                if len(v)!=5 or v[0]!=0 or not np.isfinite(v).all(): raise ValueError(f'단일 클래스 YOLO 라벨 오류: {label}')
                _,x,y,bw,bh=v
                if not (0<=x<=1 and 0<=y<=1 and 0<bw<=1 and 0<bh<=1): raise ValueError(f'좌표 오류: {label}')
                box=[(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h]
                box=np.clip(box,[0,0,0,0],[w,h,w,h]).tolist()
                if box[2]<=box[0] or box[3]<=box[1]: raise ValueError(f'빈 bbox: {label}')
                raw.append(box)
            pixel_hash=hashlib.sha256(np.asarray(im.shape,dtype=np.int64).tobytes()+im.tobytes()).hexdigest()
            if pixel_hash in fingerprints and fingerprints[pixel_hash][0]!=split:
                raise ValueError(f'split 간 동일 이미지 누수: {p}, {fingerprints[pixel_hash]}')
            fingerprints[pixel_hash]=(split,str(p))
            rows.append({'path':str(p),'gt':np.asarray(raw,dtype=np.float32).reshape(-1,4)})
            manifest.append({'split':split,'path':str(p),'label_sha256':sha(label),'pixel_sha256':pixel_hash,'objects':len(raw)})
        if sum(len(r['gt']) for r in rows)==0: raise ValueError(f'{split}: 포트홀 GT가 전혀 없습니다')
        records[split]=rows
    # Common augmented variants must stay in their original split.
    import re
    families={}
    for split,rows in records.items():
        for r in rows:
            family=re.sub(r'_envaug_\d+$','',Path(r['path']).stem)
            family=re.sub(r'\.rf\.[^.]+$','',family)
            if family in families and families[family]!=split:
                raise ValueError(f'원본/증강 family가 다른 split에 존재: {family}')
            families[family]=split
    # Optional CSV: path,group (relative to data_root). Group = source video/road/session.
    if cfg.get('groups_csv'):
        with open(cfg['groups_csv'],newline='',encoding='utf-8') as f: groups={r['path']:r['group'] for r in csv.DictReader(f)}
        used={}
        for split,rows in records.items():
            for r in rows:
                key=Path(r['path']).relative_to(root).as_posix()
                if key not in groups: raise ValueError(f'group 정보 누락: {key}')
                g=groups[key]
                if not g: raise ValueError(f'빈 group: {key}')
                if g in used and used[g]!=split: raise ValueError(f'영상/도로 group 누수: {g}')
                used[g]=split
    else:
        print('주의: 동일 영상의 인접 프레임/유사 이미지 누수는 자동 검출되지 않습니다. groups_csv로 확인하세요.')
    dump(out/'dataset_manifest.json',manifest)
    return records


class CropCNN(nn.Module):
    def __init__(self):
        super().__init__()
        blocks=[]; c=3
        for width in [32,64,128]:
            blocks += [nn.Conv2d(c,width,3,padding=1),nn.BatchNorm2d(width),nn.ReLU(),nn.MaxPool2d(2)]
            c=width
        self.net=nn.Sequential(*blocks,nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.2),nn.Linear(128,1))
    def forward(self,x): return self.net(x).flatten()


def crop_tensor(im,box,size):
    h,w=im.shape[:2]
    x1,y1=np.floor(box[:2]).astype(int); x2,y2=np.ceil(box[2:]).astype(int)
    x1,y1=max(0,min(x1,w-1)),max(0,min(y1,h-1))
    x2,y2=max(x1+1,min(x2,w)),max(y1+1,min(y2,h))
    crop=cv2.cvtColor(cv2.resize(im[y1:y2,x1:x2],(size,size)),cv2.COLOR_BGR2RGB)
    return torch.from_numpy(crop.transpose(2,0,1).copy()).float()/255.


class Crops(Dataset):
    def __init__(self,items,size,augment=False): self.items=items; self.size=size; self.augment=augment
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,box,label=self.items[i]; im=cv2.imread(p)
        if im is None: raise ValueError(p)
        x=crop_tensor(im,box,self.size)
        if self.augment:
            if random.random()<.5: x=torch.flip(x,[2])
            x=(x*random.uniform(.8,1.2)).clamp(0,1)
        return x,torch.tensor(float(label))


def predict_yolo(model,im,cfg):
    r=model.predict(im,imgsz=cfg['imgsz'],conf=cfg['proposal_conf'],iou=cfg['nms_iou'],
                    max_det=cfg['max_det'],device=cfg['device'],half=False,verbose=False)[0]
    return {'boxes':r.boxes.xyxy.cpu().numpy(),'scores':r.boxes.conf.cpu().numpy()}


def make_crops(rows,det,cfg,train):
    items=[]; rng=np.random.default_rng(cfg['seed'])
    for row in rows:
        im=cv2.imread(row['path']); h,w=im.shape[:2]; gt=row['gt']
        proposals=predict_yolo(det,im,cfg)['boxes']
        if train:
            extra=list(gt.copy())
            for b in gt:
                for _ in range(2):
                    wh=b[2:]-b[:2]; jitter=rng.uniform(-.1,.1,4)*np.tile(wh,2)
                    j=np.clip(b+jitter,[0,0,0,0],[w,h,w,h])
                    if j[2]>j[0] and j[3]>j[1]: extra.append(j)
            # Training-only backgrounds; ambiguous overlap is discarded below.
            for _ in range(cfg['backgrounds_per_image']):
                bw=rng.uniform(.05,.45)*w; bh=rng.uniform(.05,.45)*h
                x=rng.uniform(0,w-bw); y=rng.uniform(0,h-bh)
                extra.append([x,y,x+bw,y+bh])
            proposals=np.concatenate([proposals,np.asarray(extra).reshape(-1,4)])
        overlap=ious(proposals,gt).max(1) if len(gt) else np.zeros(len(proposals))
        for b,v in zip(proposals,overlap):
            if v>=.5: items.append((row['path'],b.tolist(),1))
            elif v<=.1: items.append((row['path'],b.tolist(),0))
    return items


def train_cnn(det,records,cfg,out):
    train=make_crops(records['train'],det,cfg,True)
    val=make_crops(records['val'],det,cfg,False)
    counts=[sum(x[2]==k for x in train) for k in [0,1]]
    val_counts=[sum(x[2]==k for x in val) for k in [0,1]]
    if min(counts)==0 or min(val_counts)==0:
        raise ValueError(f'CNN train/val 후보에 양성·음성이 모두 필요합니다: train={counts}, val={val_counts}. split/가중치/라벨을 확인하세요.')
    dump(out/'cnn_crop_counts.json',{'train_negative_positive':counts,'val_negative_positive':[sum(x[2]==k for x in val) for k in [0,1]],'label_rule':'positive IoU>=.5; negative IoU<=.1; otherwise ignore'})
    # Save exact training crop metadata for audit; no test crops involved.
    dump(out/'cnn_train_crops.json',[{'path':p,'box':b,'label':y} for p,b,y in train])
    generator=torch.Generator().manual_seed(cfg['seed'])
    tl=DataLoader(Crops(train,cfg['crop_size'],True),batch_size=cfg['cnn_batch'],shuffle=True,num_workers=0,generator=generator)
    vl=DataLoader(Crops(val,cfg['crop_size']),batch_size=cfg['cnn_batch'],num_workers=0)
    net=CropCNN().to(cfg['torch_device']); opt=torch.optim.AdamW(net.parameters(),lr=cfg['cnn_lr'],weight_decay=1e-4)
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(counts[0]/counts[1],device=cfg['torch_device']))
    best=float('inf'); stale=0; history=[]
    for epoch in range(cfg['cnn_epochs']):
        net.train(); tr=0.
        for x,y in tl:
            x,y=x.to(cfg['torch_device']),y.to(cfg['torch_device']); opt.zero_grad()
            loss=loss_fn(net(x),y); loss.backward(); opt.step(); tr+=float(loss.detach())*len(y)
        net.eval(); va=0.
        with torch.inference_mode():
            for x,y in vl:
                x,y=x.to(cfg['torch_device']),y.to(cfg['torch_device']); va+=float(loss_fn(net(x),y))*len(y)
        va/=len(val); history.append({'epoch':epoch+1,'train_loss':tr/len(train),'val_loss':va})
        print(f'CNN {epoch+1}: train={tr/len(train):.4f}, val={va:.4f}')
        if va<best:
            best=va; stale=0
            torch.save({'state_dict':net.state_dict(),'crop_size':cfg['crop_size'],'normalization':'RGB float32 /255'},out/'cnn_best.pt')
        else: stale+=1
        if stale>=cfg['cnn_patience']: break
    dump(out/'cnn_history.json',history)
    net.load_state_dict(torch.load(out/'cnn_best.pt',map_location=cfg['torch_device'],weights_only=True)['state_dict'])
    return net.eval()


@torch.inference_mode()
def cnn_scores(net,im,boxes,cfg):
    if len(boxes)==0: return np.empty(0,dtype=np.float32)
    result=[]
    for start in range(0,len(boxes),cfg['cnn_batch']):
        x=torch.stack([crop_tensor(im,b,cfg['crop_size']) for b in boxes[start:start+cfg['cnn_batch']]])
        result.append(net(x.to(cfg['torch_device'])).sigmoid().cpu().numpy())
    return np.concatenate(result)


def collect(det,net,rows,cfg):
    base=[]; probs=[]
    for row in rows:
        im=cv2.imread(row['path']); p=predict_yolo(det,im,cfg)
        base.append(p); probs.append(cnn_scores(net,im,p['boxes'],cfg))
    return base,probs


def rescore(base,probs,alpha):
    return [{'boxes':p['boxes'],'scores':p['scores']*np.maximum(q,1e-8)**alpha} for p,q in zip(base,probs)]


def map_metrics(pred,rows,cfg):
    metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',max_detection_thresholds=[1,10,cfg['max_det']])
    pp=[{'boxes':torch.tensor(p['boxes'],dtype=torch.float32),'scores':torch.tensor(p['scores'],dtype=torch.float32),'labels':torch.zeros(len(p['scores']),dtype=torch.int64)} for p in pred]
    tt=[{'boxes':torch.tensor(r['gt'],dtype=torch.float32),'labels':torch.zeros(len(r['gt']),dtype=torch.int64)} for r in rows]
    metric.update(pp,tt); m=metric.compute()
    return {k:float(m[k]) for k in ['map','map_50','map_75','map_small','map_medium','map_large']}


def counts_at(pred,rows,threshold,iou=.5):
    result=[]
    for p,r in zip(pred,rows):
        idx=np.flatnonzero(p['scores']>=threshold)
        idx=idx[np.argsort(-p['scores'][idx],kind='stable')]
        overlap=ious(p['boxes'][idx],r['gt']); used=set(); tp=0
        for ov in overlap:
            eligible=[j for j in range(len(ov)) if j not in used and ov[j]>=iou]
            if eligible:
                j=max(eligible,key=lambda j:ov[j]); used.add(j); tp+=1
        result.append([tp,len(idx)-tp,len(r['gt'])-tp])
    return np.asarray(result,dtype=np.int64)


def operating_metrics(counts):
    tp,fp,fn=np.asarray(counts).sum(0)
    p=tp/max(tp+fp,1); r=tp/max(tp+fn,1)
    return {'precision':float(p),'recall':float(r),'f1':float(2*tp/max(2*tp+fp+fn,1)),
            'tp':int(tp),'fp':int(fp),'fn':int(fn),'fp_per_image':float(fp/len(counts))}


def tune_threshold(pred,rows,cfg):
    grid=np.unique(np.r_[cfg['proposal_conf'],np.linspace(.01,.99,99)])
    values=[operating_metrics(counts_at(pred,rows,float(t)))['f1'] for t in grid]
    # deterministic tie-breaking: smallest threshold, favor recall.
    i=int(np.argmax(values)); return float(grid[i]),float(values[i])


def paired_bootstrap(base,hybrid,seed,n):
    rng=np.random.default_rng(seed); deltas=[]
    for _ in range(n):
        ix=rng.integers(0,len(base),len(base))
        deltas.append(operating_metrics(hybrid[ix])['f1']-operating_metrics(base[ix])['f1'])
    return {'delta_f1':operating_metrics(hybrid)['f1']-operating_metrics(base)['f1'],
            'ci95':np.quantile(deltas,[.025,.975]).tolist(),'resamples':n,'unit':'image; assumes independent images',
            'scope':'test sampling uncertainty only; not training-seed uncertainty'}


def sync(cfg):
    if cfg['torch_device'].startswith('cuda'): torch.cuda.synchronize()


def benchmark(det,net,rows,alpha,cfg):
    # Decode before timer; include preprocessing, YOLO, NMS, crop, CNN, score transfer.
    ims=[cv2.imread(r['path']) for r in rows[:cfg['benchmark_images']]]
    times={'YOLOv8':[],'YOLOv8+CNN':[]}
    def run(im,hybrid):
        p=predict_yolo(det,im,cfg)
        if hybrid:
            q=cnn_scores(net,im,p['boxes'],cfg); p['scores']=p['scores']*np.maximum(q,1e-8)**alpha
        return p
    for _ in range(3):
        for hybrid in [False,True]: run(ims[0],hybrid)
    for rep in range(cfg['benchmark_repeats']):
        order=[False,True] if rep%2==0 else [True,False]
        for im in ims:
            for hybrid in order:
                sync(cfg); start=time.perf_counter(); run(im,hybrid); sync(cfg)
                times['YOLOv8+CNN' if hybrid else 'YOLOv8'].append((time.perf_counter()-start)*1000)
    return {k:{'mean_ms':float(np.mean(v)),'median_ms':float(np.median(v)),'p95_ms':float(np.percentile(v,95)),
               'fps':float(1000/np.mean(v)),'samples':len(v),'scope':'in-memory single image; disk decoding excluded; FP32'} for k,v in times.items()}


def write_csv(path,rows):
    with open(path,'w',newline='',encoding='utf-8-sig') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)


def run(cfg):
    cfg=copy.deepcopy(cfg)
    cfg['device']=0 if torch.cuda.is_available() else 'cpu'
    cfg['torch_device']='cuda:0' if torch.cuda.is_available() else 'cpu'
    if not 0<cfg['proposal_conf']<.01: raise ValueError('mAP용 proposal_conf는 0.001 권장')
    if cfg['max_det']<=10: raise ValueError('max_det > 10 필요')
    out=Path(cfg['output_root'])/f"week5_{time.strftime('%Y%m%d_%H%M%S')}_{time.time_ns()%1000000}"
    out.mkdir(parents=True,exist_ok=False); seed_all(cfg['seed']); dump(out/'config.json',cfg)
    (out/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    dump(out/'environment.json',{'python':platform.python_version(),'torch':torch.__version__,
                                'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
    records=audit_dataset(cfg,out)
    if cfg['yolo_weights']:
        path=Path(cfg['yolo_weights'])
        if not path.exists(): raise FileNotFoundError(path)
        if not cfg['weights_provenance_confirmed']:
            raise ValueError('기존 YOLO 가중치가 test를 학습/모델 선택에 사용하지 않았는지 확인하고 weights_provenance_confirmed=True로 설정하세요')
        det=YOLO(str(path))
        if det.task!='detect' or len(det.names)!=1: raise ValueError('포트홀 단일 클래스 detect 가중치 필요')
    else:
        if any(Path(rel).name!='images' for rel in cfg['splits'].values()):
            raise ValueError('YOLO 신규 학습은 표준 split/images 및 split/labels 구조가 필요합니다. 증강 데이터를 별도 루트의 표준 폴더로 정리하세요.')
        data={'path':str(Path(cfg['data_root']).resolve()),'train':cfg['splits']['train'],'val':cfg['splits']['val'],'test':cfg['splits']['test'],'names':{0:'pothole'}}
        import yaml
        yaml_path=out/'data.yaml'; yaml_path.write_text(yaml.safe_dump(data,allow_unicode=True))
        det=YOLO(cfg['yolo_init'])
        det.train(data=str(yaml_path),epochs=cfg['yolo_epochs'],patience=20,imgsz=cfg['imgsz'],batch=cfg['yolo_batch'],
                  optimizer='AdamW',lr0=.001,lrf=.01,cos_lr=True,hsv_h=.015,hsv_s=.5,hsv_v=.4,scale=.5,
                  flipud=0.,fliplr=.5,close_mosaic=15,seed=cfg['seed'],deterministic=True,
                  device=cfg['device'],project=str(out),name='detector',exist_ok=False)
        path=Path(det.trainer.best); det=YOLO(str(path))
    dump(out/'detector_provenance.json',{'weights_path':str(path),'sha256':sha(path),
                                        'reuse':bool(cfg['yolo_weights']),'user_confirmed_no_test_training':cfg['weights_provenance_confirmed']})
    # Exact same detector instance/weights serves both methods.
    net=train_cnn(det,records,cfg,out)
    vb,vq=collect(det,net,records['val'],cfg)
    trials=[]
    for alpha in cfg['alpha_grid']:
        p=rescore(vb,vq,alpha); m=map_metrics(p,records['val'],cfg)
        trials.append({'alpha':alpha,**m})
    # alpha=0 is an explicit no-CNN-effect control. Equal mAP selects it.
    best=max(trials,key=lambda x:(x['map'],-x['alpha'])); alpha=best['alpha']
    vh=rescore(vb,vq,alpha)
    t0,f0=tune_threshold(vb,records['val'],cfg); t1,f1=tune_threshold(vh,records['val'],cfg)
    locked={'alpha':alpha,'yolo_threshold':t0,'hybrid_threshold':t1,'validation_f1':[f0,f1],
            'selection':'alpha: val COCO mAP50-95; thresholds: val F1 at IoU .5',
            'alpha_zero_means':'CNN did not improve validation mAP; no accuracy gain should be claimed'}
    dump(out/'locked_validation_settings.json',locked); write_csv(out/'validation_alpha_trials.csv',trials)
    # Test is first inferred here, after all model/threshold choices have been frozen.
    tb,tq=collect(det,net,records['test'],cfg); th=rescore(tb,tq,alpha)
    speed=benchmark(det,net,records['test'],alpha,cfg); summaries=[]; counts=[]
    for name,p,t in [('YOLOv8',tb,t0),('YOLOv8+CNN',th,t1)]:
        c=counts_at(p,records['test'],t); counts.append(c)
        summaries.append({'model':name,'threshold':t,'alpha':0 if name=='YOLOv8' else alpha,
                          **map_metrics(p,records['test'],cfg),**operating_metrics(c),**{k:v for k,v in speed[name].items() if k not in ['scope']}})
    write_csv(out/'test_comparison.csv',summaries)
    shared=[{'model':n,'threshold':cfg['shared_threshold'],**operating_metrics(counts_at(p,records['test'],cfg['shared_threshold']))}
            for n,p in [('YOLOv8',tb),('YOLOv8+CNN',th)]]
    write_csv(out/'test_shared_threshold.csv',shared)
    dump(out/'paired_bootstrap_f1.json',paired_bootstrap(*counts,cfg['seed'],cfg['bootstrap_n']))
    dump(out/'latency.json',speed)
    per_image=[]; predictions=[]
    for i,r in enumerate(records['test']):
        per_image.append({'image':r['path'],'gt':len(r['gt']),**{f'{name}_{field}':int(counts[j][i,k]) for j,name in enumerate(['yolo','hybrid']) for k,field in enumerate(['tp','fp','fn'])}})
        predictions.append({'image':r['path'],'gt':r['gt'].tolist(),'boxes':tb[i]['boxes'].tolist(),
                            'yolo_scores':tb[i]['scores'].tolist(),'cnn_probs':tq[i].tolist(),'hybrid_scores':th[i]['scores'].tolist()})
    write_csv(out/'test_per_image.csv',per_image); dump(out/'test_predictions.json',predictions)
    # Error cases selected only for reporting, after settings have been locked.
    case_dir=out/'error_cases'; case_dir.mkdir()
    worst=sorted(range(len(per_image)),key=lambda i:per_image[i]['yolo_fp']+per_image[i]['hybrid_fp']+per_image[i]['hybrid_fn'],reverse=True)[:8]
    for rank,i in enumerate(worst):
        panels=[]
        for name,p,t in [('YOLOv8',tb[i],t0),('YOLOv8+CNN',th[i],t1)]:
            im=cv2.imread(records['test'][i]['path'])
            for box in records['test'][i]['gt']:
                x1,y1,x2,y2=box.astype(int); cv2.rectangle(im,(x1,y1),(x2,y2),(0,255,0),2)
            for box,score in zip(p['boxes'],p['scores']):
                if score<t: continue
                x1,y1,x2,y2=box.astype(int); cv2.rectangle(im,(x1,y1),(x2,y2),(0,0,255),2)
                cv2.putText(im,f'{score:.2f}',(x1,max(16,y1)),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,0,255),1)
            cv2.putText(im,name,(8,24),cv2.FONT_HERSHEY_SIMPLEX,.7,(255,255,255),2)
            panels.append(im)
        cv2.imwrite(str(case_dir/f'{rank:02d}.jpg'),np.concatenate(panels,axis=1))
    import matplotlib.pyplot as plt
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    metrics=['map_50','map','precision','recall','f1']; x=np.arange(len(metrics))
    for j,r in enumerate(summaries): axs[0].bar(x+(j-.5)*.35,[r[k] for k in metrics],width=.35,label=r['model'])
    axs[0].set_xticks(x,metrics); axs[0].set_ylim(0,1); axs[0].legend(); axs[0].set_title('Held-out test: detection metrics')
    axs[1].bar([r['model'] for r in summaries],[r['mean_ms'] for r in summaries]); axs[1].set_ylabel('ms / image'); axs[1].set_title('YOLO + crop + CNN; disk decode excluded')
    fig.tight_layout(); fig.savefig(out/'comparison.png',dpi=180); plt.close(fig)
    for r in summaries: print(r)
    print('선택된 alpha:',alpha,'| 결과 폴더:',out)
    return out,summaries


## 3. 학습·검증 선택·최종 테스트
YOLO 가중치가 비어 있으면 YOLO도 학습합니다. CNN 학습 후 val에서 alpha와 threshold를 고정하고 test를 평가합니다. test 결과를 보고 설정을 반복 변경하지 마세요.


In [ ]:
import sys
if "/content" not in sys.path: sys.path.insert(0,"/content")
from week5_compare import run
OUT, RESULTS = run(CFG)


## 4. 비교표 및 그래프
0~1 지표입니다. 수치 차이를 보고서에 %p로 표현할 때 100을 곱하세요. mAP 평가 설정은 양쪽 모두 최대 300 detection/이미지입니다.


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv(OUT/"test_comparison.csv"))
display(pd.read_csv(OUT/"test_shared_threshold.csv"))
display(Image(filename=str(OUT/"comparison.png")))
import json
print("Paired ΔF1 CI:", json.loads((OUT/"paired_bootstrap_f1.json").read_text()))
print("설정:", json.loads((OUT/"locked_validation_settings.json").read_text()))


## 5. 오류 사례 확인
녹색은 GT, 빨간색은 선택된 운영 임계값 이상 탐지입니다. 왼쪽 YOLO, 오른쪽 YOLO+CNN. CNN은 YOLO가 놓친 새로운 박스를 만들 수 없습니다.


In [ ]:
for case in sorted((OUT/"error_cases").glob("*.jpg"))[:4]:
    display(Image(filename=str(case), width=1100))


## 보고서 해석
- mAP50–95, F1, Recall, FP/image, latency를 함께 비교하세요. CNN의 FP 감소와 FN 증가를 모두 확인합니다.
- alpha=0이면 val에서 CNN 재평가의 이점이 확인되지 않았다는 결과입니다.
- ΔF1 신뢰구간은 고정 모델의 test 이미지 표본 불확실성이며 학습 seed 변동이나 mAP 유의성을 뜻하지 않습니다.
- 동일 영상의 인접 프레임은 촬영 group별 분할과 group bootstrap이 필요합니다.
- seed 42/43/44 반복 시 test로 최적 seed를 고르지 말고 평균±표준편차를 보고하세요.
- 설치 버전 및 환경은 결과 폴더에 기록됩니다. 자세한 제한과 파일 설명은 동봉 README.md를 확인하세요.
